# Bond & bond-future analytics tutorial

This notebook walks through the **cqfi** Python API for cash-bond, CMT, and bond-future
basis analytics. It has two parallel tracks:

1. **Registry / prepopulated objects** — `IssuerProfile`s in `ISSUERS`, contracts in
   `BOND_FUTURE_CONVENTIONS`, bonds from `BondManager` / `bond_universe` when the DB is available.
2. **User-built objects** — construct your own `IssuerProfile`, `Bond`s, `BondFutureConvention`,
   and `DeliveryBasket`, register the issuer temporarily, then run the same calculators.

### Requirements

- Project installed editable: `uv sync` from the repo root.
- Kernel: select the project `.venv` (or `uv run jupyter lab`).
- **Optional databases** (configured in `config/cqfi.yaml`):
  - `ycs_db` — live yield curves via `QuantlibMarketContextManager`
  - `bond_analytics_db` — `BondManager` lookups and `DeliveryBasket.auto`

When those DBs are missing, the notebook falls back to **in-memory bonds** and a **flat discount curve**,
so the futures sections still run end-to-end.


## 0. Setup

Add the project `src` tree to `sys.path` when the kernel was not started via `uv run`, then load settings.


In [ ]:
from __future__ import annotations

import sys
from datetime import date
from pathlib import Path

import QuantLib as ql

# Repo root = parent of notebooks/
ROOT = Path.cwd().resolve()
if (ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = ROOT
elif (ROOT.parent / "pyproject.toml").exists():
    PROJECT_ROOT = ROOT.parent
else:
    PROJECT_ROOT = ROOT
src = PROJECT_ROOT / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from cqfi.config import load_settings
from cqfi.issuers import ISSUERS, IssuerProfile, RateType, RepoMarket, resolve_issuer
from cqfi.instruments import Bond
from cqfi.bond_manager import BondManager
from cqfi.analytics_input import BondAnalyticsInput, CmtAnalyticsInput
from cqfi.numeric_term_structure import NumericTermStructure
from cqfi.quantlib.quantlib_analytics_calculator import QuantLibAnalyticsCalculator
from cqfi.quantlib.quantlib_market_context_manager import QuantlibMarketContextManager
from cqfi.quantlib.quantlib_market_context import (
    QuantLibCurveCollection,
    QuantlibMarketContext,
)
from cqfi.date_utils import to_ql_date
from cqfi.bond_futures import (
    BOND_FUTURE_CONVENTIONS,
    BondFuture,
    BondFutureConvention,
    BasketRestrictions,
    ConversionFactorMethod,
    MaturityRange,
    months,
    resolve_bond_future_convention,
)
from cqfi.day_of_month import DayOfMonthSpec
from cqfi.delivery_basket import DeliveryBasket
from cqfi.bond_future_input import BondFutureInput
from cqfi.quantlib.quantlib_bond_future_calculator import QuantLibBondFutureCalculator
from cqfi.quantlib.quantlib_conversion_factor import conversion_factor

settings = load_settings(PROJECT_ROOT / "config" / "cqfi.yaml")
print("Project root:", PROJECT_ROOT)
print("ycs_db:", settings.ycs_db_path, "exists=", settings.ycs_db_path.exists())
print(
    "bond_analytics_db:",
    settings.bond_analytics_db_path,
    "exists=",
    settings.bond_analytics_db_path.exists(),
)


### Helper: flat market context

Used whenever live curves from `ycs_db` are unavailable (or for fully offline demos).


In [ ]:
# Separate label so flat demos do not clash with live BOND_ZERO options.
FLAT_CURVE_LABEL = "FLAT_DEMO"

def flat_market_context(
    as_of: date,
    issuer: str,
    rate: float = 0.03,
    *,
    curve_label: str = FLAT_CURVE_LABEL,
) -> QuantlibMarketContext:
    """Build a QuantlibMarketContext with a single flat zero curve."""
    ql.Settings.instance().evaluationDate = to_ql_date(as_of)
    profile = ISSUERS[issuer]
    curve = ql.YieldTermStructureHandle(
        ql.FlatForward(
            to_ql_date(as_of),
            rate,
            profile.day_count,
            ql.Compounded,
            profile.frequency,
        )
    )
    collection = QuantLibCurveCollection(as_of)
    collection.set_bond_curve(issuer, curve)
    ctx = QuantlibMarketContext()
    ctx.set_curve_collection(collection, label=curve_label)
    return ctx


def resolve_market_context(
    as_of: date, issuer: str, *, flat_rate: float = 0.03
) -> tuple[QuantlibMarketContext, str, str]:
    """Prefer live ycs curves; fall back to a flat curve.

    Returns ``(context, source_label, curve_label)``.
    """
    mgr = QuantlibMarketContextManager.instance()
    handle = mgr.get(as_of, issuer, "BOND_ZERO")
    if handle is not None:
        ctx = mgr.get(as_of)
        if ctx is not None:
            return ctx, "live (ycs_db)", "BOND_ZERO"
    return (
        flat_market_context(as_of, issuer, flat_rate),
        "flat fallback",
        FLAT_CURVE_LABEL,
    )


---
## Part A — Prepopulated / registry objects

### A1. Issuer profiles

`ISSUERS` is the static registry of sovereign convention packs. `resolve_issuer` accepts
canonical codes and common aliases (e.g. `"BTP"` → Italy).


In [ ]:
print(f"{len(ISSUERS)} issuers:", ", ".join(sorted(ISSUERS)))

ita = resolve_issuer("BTP")  # alias → ITA
deu = ISSUERS["DEU"]
print(ita.source_code, ita.name, ita.currency)
print("ITA settlement days:", ita.settlement_days)
print("ITA day count:", ita.day_count.name())
print("DEU settlement on 2026-05-15:", deu.settlement_date(date(2026, 5, 15)))


### A2. Bonds from the universe (or in-memory stand-ins)

`BondManager` reads `bond_universe` in `bond_analytics_db`. If that DB is empty or missing,
we construct a few Italian BTPs by hand — enough for futures basket demos.


In [ ]:
TRADE_DATE = date(2026, 5, 15)
BOND_TRADE_DATE = date(2024, 1, 15)  # typical live-curve demo date

manager = BondManager.instance()

# Always available in-memory deliverables for FBTP Sep-2026 (8y6m–11y remaining).
REGISTRY_BONDS = [
    Bond(
        issuer="ITA",
        maturity=date(2035, 4, 1),
        coupon=2.50,
        bond_id="IT203504",
        user_friendly_id="ita203504",
        issue_date=date(2015, 4, 1),
    ),
    Bond(
        issuer="ITA",
        maturity=date(2036, 2, 1),
        coupon=4.00,
        bond_id="IT203602",
        user_friendly_id="ita203602",
        issue_date=date(2015, 4, 1),
    ),
    Bond(
        issuer="ITA",
        maturity=date(2037, 8, 1),
        coupon=3.25,
        bond_id="IT203708",
        user_friendly_id="ita203708",
        issue_date=date(2015, 4, 1),
    ),
]

def _pick_analytics_bond() -> Bond:
    """Prefer a long coupon bond from bond_universe; else in-memory stand-in."""
    try:
        candidates = manager.get_by_issuer("ITA")
    except Exception:
        candidates = []
    for bond in candidates:
        if (
            bond.coupon
            and bond.coupon > 0
            and bond.maturity
            and bond.maturity >= date(2030, 1, 1)
            and bond.issue_date
            and bond.issue_date < BOND_TRADE_DATE
        ):
            return bond
    return REGISTRY_BONDS[0]

sample_from_db = _pick_analytics_bond()
print("Bond for cash analytics:")
print(sample_from_db.as_dict())
for b in REGISTRY_BONDS:
    print(b.user_friendly_id, b.coupon, b.maturity)


### A3. Cash-bond analytics (registry issuer + bond)

`BondAnalyticsInput.from_bond` builds the request. `QuantLibAnalyticsCalculator.compute_bond_analytics`
returns `(bond_metrics, mm_cmt, mm_fc_cmt)` — the cash bond plus maturity-matched par and
fixed-coupon CMT comparables.


In [ ]:
bond_for_analytics = sample_from_db
analytics_date = BOND_TRADE_DATE
issuer_code = bond_for_analytics.issuer

market, market_source, curve_label = resolve_market_context(
    analytics_date, issuer_code
)
print(
    "Market context:", market_source, "curve=", curve_label,
    "for", issuer_code, "on", analytics_date,
)

repo = NumericTermStructure(
    {"1m": 3.0, "3m": 3.0, "6m": 3.0, "1y": 3.0},
    as_of=analytics_date,
)
request = BondAnalyticsInput.from_bond(
    bond_for_analytics,
    trade_date=analytics_date,
    repo_term_structure=repo,
)

calc = QuantLibAnalyticsCalculator()
try:
    bond_m, mm_cmt, mm_fc = calc.compute_bond_analytics(
        request, market, curve_label=curve_label
    )
except Exception as exc:
    print("Live/DB bond failed (", type(exc).__name__, "); retry with in-memory BTP")
    bond_for_analytics = REGISTRY_BONDS[0]
    market, market_source, curve_label = resolve_market_context(
        analytics_date, bond_for_analytics.issuer
    )
    request = BondAnalyticsInput.from_bond(
        bond_for_analytics,
        trade_date=analytics_date,
        repo_term_structure=repo,
    )
    bond_m, mm_cmt, mm_fc = calc.compute_bond_analytics(
        request, market, curve_label=curve_label
    )

print("Bond:", bond_for_analytics.user_friendly_id or bond_for_analytics.bond_id)
print("YTM %:", bond_m.yield_to_maturity)
print("Clean:", bond_m.clean_price, "Dirty:", bond_m.dirty_price)
print("Duration / Convexity:", bond_m.duration, bond_m.convexity)
print("Z-spread (bps):", bond_m.z_spread)
if mm_cmt is not None:
    print("MM par CMT clean / YTM:", mm_cmt.clean_price, mm_cmt.yield_to_maturity)
if mm_fc is not None:
    print("MM fixed-coupon CMT clean / YTM:", mm_fc.clean_price, mm_fc.yield_to_maturity)


### A4. CMT analytics (registry issuer)

Forward-starting CMTs use `CmtAnalyticsInput.from_string` with a composite tenor
(e.g. `10y`, `10y2y`).


In [ ]:
cmt_date = BOND_TRADE_DATE
cmt_market, cmt_src, cmt_curve = resolve_market_context(cmt_date, "DEU")
print("CMT market:", cmt_src, "curve=", cmt_curve)

cmt_request = CmtAnalyticsInput.from_string("DEU", "10y", trade_date=cmt_date)
cmt_metrics = QuantLibAnalyticsCalculator().compute_cmt_analytics(
    cmt_request, cmt_market, curve_label=cmt_curve
)
print("DEU 10y CMT clean (approx 100 at par):", cmt_metrics.clean_price)
print("YTM / par yield:", cmt_metrics.yield_to_maturity, cmt_metrics.par_yield)
print("Duration:", cmt_metrics.duration)


### A5. Bond-future conventions (registry)

`BOND_FUTURE_CONVENTIONS` keys are canonical exchange codes. `resolve_bond_future_convention`
also accepts Bloomberg roots and synonyms (`"IK"` → FBTP). `BondFuture.parse` attaches a delivery month.


In [ ]:
print("Sample conventions:", sorted(BOND_FUTURE_CONVENTIONS)[:12], "...")

fbtp = resolve_bond_future_convention("IK")  # Bloomberg root → FBTP
print(fbtp.name, fbtp.exchange, fbtp.issuer_code, fbtp.notional_coupon)
print("CF method:", fbtp.conversion_factor_method)
print("Restrictions:", fbtp.restrictions.as_dict())

future = BondFuture.parse("IKU6", today=date(2026, 5, 1))  # Sep-2026
print("Contract:", future)
print("Delivery end:", future.delivery_end_date())
print("Reference date:", future.reference_date())


### A6. Delivery basket from registry convention

- Prefer `DeliveryBasket.auto` when `bond_analytics_db` has a populated `bond_universe`.
- Otherwise build the basket with `.add(...)` using the in-memory BTPs from A2.


In [ ]:
future_u6 = BondFuture(BOND_FUTURE_CONVENTIONS["FBTP"], 9, 2026)
basket = None
basket_source = None

try:
    auto = DeliveryBasket.auto(future_u6, name="fbtp_auto")
    if len(auto) > 0:
        basket = auto
        basket_source = f"DeliveryBasket.auto ({len(auto)} bonds from bond_universe)"
except Exception as exc:
    print("auto() unavailable:", type(exc).__name__, exc)

if basket is None:
    basket = DeliveryBasket(bond_future=future_u6, name="fbtp_manual")
    for bond in REGISTRY_BONDS:
        basket.add(bond)
    basket_source = f"manual add ({len(basket)} in-memory bonds)"

print(basket_source)
first = basket.bonds()[0]
print("CF", first.user_friendly_id, conversion_factor(first, future_u6))
basket.to_polars()


### A7. Bond-future basis analytics (registry path)

Pass an optional basket-wide repo curve on `BondFutureInput`. Omit `futures_price` to imply
the price that zeros the CTD net basis. Results are ranked cheapest-to-deliver first.


In [ ]:
fut_market, fut_src, fut_curve = resolve_market_context(
    TRADE_DATE, "ITA", flat_rate=0.03
)
print("Futures market:", fut_src, "curve=", fut_curve)

basket_repo = NumericTermStructure({"1d": 3.0}, as_of=TRADE_DATE)
basket.set_repo_term_structure(
    basket.bonds()[0],
    NumericTermStructure({"3m": 1.0, "1y": 1.2}, as_of=TRADE_DATE),
)

fut_request = BondFutureInput.from_basket(
    basket,
    TRADE_DATE,
    repo_term_structure=basket_repo,
    curve_label=fut_curve,
)
fut_result = QuantLibBondFutureCalculator().compute_bond_future_analytics(
    fut_request, fut_market, curve_label=fut_curve
)

ctd = fut_result.ctd()
print("Contract:", fut_result.bond_future)
print("Futures price:", fut_result.futures_price, "implied=", fut_result.futures_price_is_implied)
print("Basket repo %:", fut_result.repo_rate)
print("CTD:", ctd.bond.user_friendly_id, "net basis", ctd.net_basis, "IRR", ctd.implied_repo_rate)
fut_result.to_polars()


---
## Part B — User-created objects

Construct a custom issuer, bonds, and future convention from scratch, then repeat the analytics.

**Important:** `Bond` and `BondFutureConvention` require `issuer` / `issuer_code` to be a key of
`ISSUERS`. Register a custom `IssuerProfile` into that dict before building bonds or conventions.
Clean up afterwards so you do not leak demo issuers into the process.


### B1. Custom `IssuerProfile`

Clone Italy-like conventions under a demo code `XIT`, then register it.


In [ ]:
CUSTOM_CODE = "XIT"

_previous_custom = ISSUERS.get(CUSTOM_CODE)

custom_issuer = IssuerProfile(
    source_code=CUSTOM_CODE,
    name="Demo Italy-like Sovereign",
    currency="EUR",
    calendar_factory=lambda: ql.TARGET(),
    day_count=ql.ActualActual(ql.ActualActual.ISDA),
    settlement_days=2,
    frequency=ql.Semiannual,
    default_rate_type=RateType.ZERO,
    payment_convention=ql.ModifiedFollowing,
    repo_day_count=ql.Actual360(),
    repo_settlement_days=0,
)
ISSUERS[CUSTOM_CODE] = custom_issuer

print(resolve_issuer(CUSTOM_CODE).name)
print("Settlement on", TRADE_DATE, "->", custom_issuer.settlement_date(TRADE_DATE))
print("Repo conventions:", custom_issuer.repo_conventions(RepoMarket.INTERNATIONAL))


### B2. Custom bonds

User-built `Bond` instances — no `bond_universe` required. Coupon, maturity, and issue date
must satisfy the custom future's basket restrictions (defined next).


In [ ]:
custom_bonds = [
    Bond(
        issuer=CUSTOM_CODE,
        maturity=date(2035, 6, 1),
        coupon=2.75,
        bond_id="XIT203506",
        user_friendly_id="xit203506",
        currency="EUR",
        issue_date=date(2016, 6, 1),
        is_green=False,
    ),
    Bond(
        issuer=CUSTOM_CODE,
        maturity=date(2036, 3, 1),
        coupon=3.50,
        bond_id="XIT203603",
        user_friendly_id="xit203603",
        currency="EUR",
        issue_date=date(2016, 3, 1),
    ),
    Bond(
        issuer=CUSTOM_CODE,
        maturity=date(2037, 9, 1),
        coupon=4.25,
        bond_id="XIT203709",
        user_friendly_id="xit203709",
        currency="EUR",
        issue_date=date(2017, 9, 1),
        issue_amount=5_000_000_000.0,
    ),
]
for b in custom_bonds:
    print(b.as_dict())


### B3. Custom `BondFutureConvention` + dated `BondFuture`

Day-of-month specs use the same grammar as the registry (`C10 1BD`, `L0 -1BD`, …).
`BasketRestrictions` declare eligibility windows in whole months.


In [ ]:
custom_convention = BondFutureConvention(
    name="XFBT",
    exchange="EUREX",
    issuer_code=CUSTOM_CODE,
    notional_maturity_years=10.0,
    notional_coupon=6.0,
    contract_size=100_000.0,
    reference_day=DayOfMonthSpec.from_string("C10 1BD"),
    delivery_start=DayOfMonthSpec.from_string("C10 1BD"),
    delivery_end=DayOfMonthSpec.from_string("C10 1BD"),
    conversion_factor_method=ConversionFactorMethod.EUREX,
    repo_market=RepoMarket.INTERNATIONAL,
    cf_decimals=6,
    bloomberg_root="XQ",
    synonyms=("XBT",),
    restrictions=BasketRestrictions(
        remaining_maturity=MaturityRange(months(8, 6), months(11)),
        exclude_green=True,
    ),
)

custom_future = BondFuture(custom_convention, delivery_month=9, delivery_year=2026)
print(custom_future)
print(custom_convention.as_json(indent=2))
print("Delivery end:", custom_future.delivery_end_date())

for bond in custom_bonds:
    ok, reason = custom_convention.restrictions.admits(
        bond, custom_future.delivery_end_date()
    )
    print(bond.user_friendly_id, "admitted=" + str(ok), reason or "")


### B4. Custom delivery basket + conversion factors

Hard-code a conversion factor on one bond to override the exchange formula.


In [ ]:
custom_basket = DeliveryBasket(bond_future=custom_future, name="custom_demo")
custom_basket.add(custom_bonds[0])
custom_basket.add(custom_bonds[1], conversion_factor=1.05)  # override CF
custom_basket.add(custom_bonds[2])

for member in custom_basket.members:
    bond = member.bond
    cf = member.conversion_factor_override
    if cf is None:
        cf = conversion_factor(bond, custom_future)
    print(bond.user_friendly_id, "CF", cf, "override", member.conversion_factor_override)

custom_basket.to_polars()


### B5. Bond analytics on a user-created bond

Same calculator as Part A — only the issuer code and market curve label change.


In [ ]:
custom_market = flat_market_context(TRADE_DATE, CUSTOM_CODE, 0.03)

custom_bond_request = BondAnalyticsInput.from_bond(
    custom_bonds[1],
    trade_date=TRADE_DATE,
    repo_term_structure=NumericTermStructure(
        {"1m": 2.8, "3m": 2.9, "6m": 3.0, "1y": 3.1},
        as_of=TRADE_DATE,
    ),
)
custom_bond_m, custom_mm, custom_mm_fc = QuantLibAnalyticsCalculator().compute_bond_analytics(
    custom_bond_request, custom_market, curve_label=FLAT_CURVE_LABEL
)
print(
    "Custom bond YTM / clean / z-spread:",
    custom_bond_m.yield_to_maturity,
    custom_bond_m.clean_price,
    custom_bond_m.z_spread,
)
print("Carry 3m:", custom_bond_m.carry_3m)


### B6. Bond-future analytics on the user-created convention

Per-bond repo override + basket-wide repo, implied futures price, CTD ranking.


In [ ]:
custom_basket.set_repo_term_structure(
    custom_bonds[0],
    NumericTermStructure({"3m": 0.5, "1y": 0.8}, as_of=TRADE_DATE),
)

custom_fut_request = BondFutureInput.from_basket(
    custom_basket,
    TRADE_DATE,
    repo_term_structure=NumericTermStructure({"1d": 3.0}, as_of=TRADE_DATE),
    curve_label=FLAT_CURVE_LABEL,
)
custom_fut_result = QuantLibBondFutureCalculator().compute_bond_future_analytics(
    custom_fut_request, custom_market, curve_label=FLAT_CURVE_LABEL
)

print("Custom contract:", custom_fut_result.bond_future)
print("F=", custom_fut_result.futures_price, "implied=", custom_fut_result.futures_price_is_implied)
ctd_c = custom_fut_result.ctd()
print(
    "CTD:",
    ctd_c.bond.user_friendly_id,
    "net_basis",
    round(ctd_c.net_basis, 6),
    "IRR",
    ctd_c.implied_repo_rate,
)
custom_fut_result.to_polars()


### B7. Observed futures price (instead of implied)

Supply `futures_price` to measure basis against a market quote.


In [ ]:
quoted = BondFutureInput.from_basket(
    custom_basket,
    TRADE_DATE,
    repo_term_structure=NumericTermStructure({"1d": 3.0}, as_of=TRADE_DATE),
    futures_price=120.0,
    curve_label=FLAT_CURVE_LABEL,
)
quoted_result = QuantLibBondFutureCalculator().compute_bond_future_analytics(
    quoted, custom_market, curve_label=FLAT_CURVE_LABEL
)
print("Implied?", quoted_result.futures_price_is_implied, "F=", quoted_result.futures_price)
quoted_result.to_polars().select(
    ["maturity", "coupon", "conversion_factor", "implied_repo_rate", "net_basis", "gross_basis", "index"]
)


### B8. Cleanup

Remove the demo issuer from `ISSUERS` so later REPL / CLI sessions are unaffected.


In [ ]:
if _previous_custom is None:
    ISSUERS.pop(CUSTOM_CODE, None)
else:
    ISSUERS[CUSTOM_CODE] = _previous_custom
print(CUSTOM_CODE, "still registered?", CUSTOM_CODE in ISSUERS)


---
## Recap

| Step | Registry path | User-built path |
|------|---------------|-----------------|
| Issuer | `ISSUERS` / `resolve_issuer` | `IssuerProfile(...)` then `ISSUERS[code] = ...` |
| Bond | `BondManager.get` / `bond_universe` | `Bond(...)` |
| Market | `QuantlibMarketContextManager` + `ycs_db` | `flat_market_context` or live curves |
| Cash analytics | `BondAnalyticsInput` + `QuantLibAnalyticsCalculator` | same |
| CMT | `CmtAnalyticsInput.from_string` | same (needs registered issuer) |
| Future convention | `BOND_FUTURE_CONVENTIONS` / `resolve_*` | `BondFutureConvention(...)` |
| Basket | `DeliveryBasket.auto` or `.add` | `.add` on a custom `BondFuture` |
| Basis analytics | `BondFutureInput` + `QuantLibBondFutureCalculator` | same |

CLI counterparts: `/bond`, `/calc`, `/dlv`, `/fut`, `/mctx`. See also
`README.md` sections **Bond analytics (Python API)** and **Bond futures (Python API)**.
